# Credit Card Fraud Detection v2
XGBoost + Deep Learning (DNN) + Hyperparameter Tuning + EDA lengkap + Export untuk REST API.

**Cara pakai di Colab:** jalankan sel dari atas ke bawah. Di sel 1 kamu akan diminta upload `creditcard.csv`.
Tunggu sampai upload selesai 100% (spinner hilang) sebelum lanjut.

## 1. Setup & upload data

In [ ]:
import json, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
OUT = Path("artifacts_v2"); OUT.mkdir(exist_ok=True)
FIG = Path("figures"); FIG.mkdir(exist_ok=True)

def savefig(name):
    plt.tight_layout(); plt.savefig(FIG / name, dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# Upload creditcard.csv (Colab). Kalau file sudah ada di folder, bagian upload dilewati.
import os
if not os.path.exists("creditcard.csv"):
    from google.colab import files
    files.upload()
df = pd.read_csv("creditcard.csv")
print(df.shape)

In [ ]:
# CEK INTEGRITAS: pastikan file tidak terpotong waktu upload.
assert df.shape == (284807, 31), f"Jumlah baris/kolom salah: {df.shape}. Hapus file & upload ulang (tunggu sampai selesai)."
assert int(df["Class"].sum()) == 492, "Jumlah fraud harus 492. File kemungkinan rusak/terpotong."
print("OK: 284.807 baris, 492 fraud")

## 2. EDA (Exploratory Data Analysis)
Tujuan: memahami data sebelum membuat model. Kita lihat bentuk data, nilai kosong, duplikat, ketimpangan kelas, dan fitur mana yang paling beda antara fraud dan normal.

In [ ]:
display(df.head())
display(df.describe().T[["mean", "std", "min", "max"]].round(3))
print("Nilai kosong total :", int(df.isna().sum().sum()))
print("Baris duplikat     :", int(df.duplicated().sum()))
print("Tipe data          :", df.dtypes.value_counts().to_dict())

In [ ]:
# Distribusi kelas (skala log supaya fraud yang sedikit tetap terlihat)
vc = df["Class"].value_counts().sort_index()
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(x=["Normal", "Fraud"], y=vc.values, ax=ax[0], palette=["#2dd4bf", "#f87171"])
ax[0].set_yscale("log"); ax[0].set_title("Distribusi kelas (skala log)")
for i, v in enumerate(vc.values): ax[0].text(i, v, f"{v:,}", ha="center", va="bottom")
ax[1].pie(vc.values, labels=["Normal", "Fraud"], autopct="%.2f%%", colors=["#2dd4bf", "#f87171"], startangle=90)
ax[1].set_title("Proporsi kelas")
savefig("01_class_distribution.png")
print(f"Fraud hanya {vc[1]/vc.sum():.3%} dari data -> data sangat tidak seimbang (imbalanced).")

In [ ]:
# Amount & Time: fraud vs normal
fig, ax = plt.subplots(2, 2, figsize=(12, 8))
for c, name, col in [(0, "Normal", "#2dd4bf"), (1, "Fraud", "#f87171")]:
    sub = df[df.Class == c]
    ax[0, 0].hist(np.log1p(sub.Amount), bins=50, alpha=.6, density=True, label=name, color=col)
    ax[0, 1].hist(sub.Time / 3600, bins=48, alpha=.6, density=True, label=name, color=col)
ax[0, 0].set_title("Distribusi log(1+Amount)"); ax[0, 0].legend()
ax[0, 1].set_title("Distribusi Time (jam sejak transaksi pertama)"); ax[0, 1].legend()
sns.boxplot(data=df, x="Class", y="Amount", ax=ax[1, 0], showfliers=False, palette=["#2dd4bf", "#f87171"])
ax[1, 0].set_title("Amount per kelas (tanpa outlier)")
print(df.groupby("Class")["Amount"].describe().round(2))
ax[1, 1].axis("off")
ax[1, 1].text(0, .5, "Jumlah transaksi fraud cenderung kecil,\nsehingga Amount saja tidak cukup\nuntuk mendeteksi fraud.", fontsize=12)
savefig("02_amount_time.png")

In [ ]:
# Fitur PCA mana yang paling membedakan fraud? (korelasi dengan Class + boxplot top fitur)
corr = df.corr()["Class"].drop("Class").sort_values(key=abs, ascending=False)
top = corr.head(8).index.tolist()
print(corr.head(8).round(3))
fig, ax = plt.subplots(2, 4, figsize=(16, 7))
for a, f in zip(ax.ravel(), top):
    sns.boxplot(data=df, x="Class", y=f, ax=a, showfliers=False, palette=["#2dd4bf", "#f87171"])
    a.set_title(f"{f} (corr={corr[f]:.2f})")
savefig("03_top_features_boxplot.png")

In [ ]:
plt.figure(figsize=(7, 8))
corr.sort_values().plot.barh(color=["#f87171" if v > 0 else "#2dd4bf" for v in corr.sort_values()])
plt.title("Korelasi tiap fitur dengan Class"); plt.xlabel("Korelasi")
savefig("04_correlation_with_class.png")

## 3. Preprocessing
Langkah:
1. Hapus baris duplikat.
2. Bagi data **dulu** (stratified): train 64% / validation 16% / test 20%. Test tidak boleh disentuh sampai evaluasi akhir.
3. Standardisasi `Amount` dan `Time` dengan **scaler terpisah**, hanya dari data *train* (supaya tidak ada kebocoran data / data leakage).

In [ ]:
from sklearn.model_selection import train_test_split
n0 = len(df)
data = df.drop_duplicates().reset_index(drop=True)
print(f"Duplikat dibuang: {n0 - len(data):,} baris -> {len(data):,} baris tersisa, fraud = {int(data.Class.sum())}")

X = data.drop(columns="Class"); y = data["Class"]
X_tr_full, X_te, y_tr_full, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
X_tr, X_va, y_tr, y_va = train_test_split(X_tr_full, y_tr_full, test_size=0.20, stratify=y_tr_full, random_state=SEED)
for n, a in [("train", y_tr), ("val", y_va), ("test", y_te)]:
    print(f"{n:5s}: {len(a):>7,} baris, fraud {int(a.sum()):>3} ({a.mean():.3%})")

In [ ]:
from sklearn.preprocessing import StandardScaler
amount_sc, time_sc = StandardScaler(), StandardScaler()      # scaler TERPISAH
amount_sc.fit(X_tr[["Amount"]]); time_sc.fit(X_tr[["Time"]])  # fit hanya dari train
V = [f"V{i}" for i in range(1, 29)]
FEATURES = ["scaled_amount", "scaled_time"] + V

def prep(d):
    out = pd.DataFrame(index=d.index)
    out["scaled_amount"] = amount_sc.transform(d[["Amount"]]).ravel()
    out["scaled_time"] = time_sc.transform(d[["Time"]]).ravel()
    for c in V: out[c] = d[c].values
    return out[FEATURES]

Xtr, Xva, Xte = prep(X_tr), prep(X_va), prep(X_te)
spw = float((y_tr == 0).sum() / (y_tr == 1).sum())
print("Bentuk fitur:", Xtr.shape, "| scale_pos_weight =", round(spw, 2))

preproc = {
    "amount": {"mean": float(amount_sc.mean_[0]), "scale": float(amount_sc.scale_[0]), "log1p": False},
    "time": {"mean": float(time_sc.mean_[0]), "scale": float(time_sc.scale_[0])},
}
print(preproc)

## 4. Fungsi evaluasi
Karena fraud hanya ~0,17%, **akurasi tidak berguna** (menebak semua "normal" sudah 99,8%). Kita pakai Precision, Recall, F1, dan **PR-AUC** (Average Precision).

In [ ]:
from sklearn.metrics import (precision_score, recall_score, f1_score, average_precision_score,
                             roc_auc_score, confusion_matrix, precision_recall_curve, roc_curve)

def best_threshold(y_true, proba):
    """Cari threshold dengan F1 terbaik (dipakai pada data VALIDATION, bukan test)."""
    p, r, t = precision_recall_curve(y_true, proba)
    f1 = 2 * p[:-1] * r[:-1] / np.clip(p[:-1] + r[:-1], 1e-12, None)
    return float(t[int(np.nanargmax(f1))])

def evaluate(name, y_true, proba, thr):
    pred = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    return {"model": name, "threshold": round(thr, 4),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred),
            "f1": f1_score(y_true, pred),
            "pr_auc": average_precision_score(y_true, proba),
            "roc_auc": roc_auc_score(y_true, proba),
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}

## 5. Model 1: XGBoost baseline (default, 100 pohon)
Ini pembanding awal. Setelah itu kita lakukan tuning dan lihat apakah benar-benar lebih baik.

In [ ]:
import xgboost as xgb
base = xgb.XGBClassifier(n_estimators=100, scale_pos_weight=spw, eval_metric="aucpr",
                         random_state=SEED, n_jobs=-1)
base.fit(Xtr, y_tr)
p_base_va, p_base_te = base.predict_proba(Xva)[:, 1], base.predict_proba(Xte)[:, 1]
print("XGBoost baseline OK")

## 6. Model 2: XGBoost + Hyperparameter Tuning
Memakai **RandomizedSearchCV** dengan **StratifiedKFold 3-fold**, metrik **average_precision (PR-AUC)**.
Parameter yang dicari: jumlah pohon, kedalaman, learning rate, subsample, colsample, min_child_weight, gamma, reg_lambda.
(Butuh beberapa menit. Aktifkan GPU di Colab tidak wajib.)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
param_dist = {
    "n_estimators": [200, 300, 500],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.03, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.5],
    "reg_lambda": [1, 5, 10],
}
search = RandomizedSearchCV(
    xgb.XGBClassifier(scale_pos_weight=spw, eval_metric="aucpr", random_state=SEED, n_jobs=-1),
    param_distributions=param_dist, n_iter=20, scoring="average_precision",
    cv=StratifiedKFold(3, shuffle=True, random_state=SEED), random_state=SEED, n_jobs=1, verbose=1, refit=True)
t0 = time.time(); search.fit(Xtr, y_tr)
print(f"Selesai {time.time()-t0:.0f} detik")
print("Parameter terbaik:", search.best_params_)
print("PR-AUC CV terbaik :", round(search.best_score_, 4))
xgb_tuned = search.best_estimator_
p_xt_va, p_xt_te = xgb_tuned.predict_proba(Xva)[:, 1], xgb_tuned.predict_proba(Xte)[:, 1]
cv = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
display(cv[["rank_test_score", "mean_test_score", "std_test_score"] + [f"param_{k}" for k in param_dist]].head(8).round(4))

In [ ]:
# Visual hasil tuning: sebaran skor semua kombinasi yang dicoba
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(cv) + 1), cv["mean_test_score"].values, "o-", color="#2dd4bf")
plt.xlabel("Peringkat kombinasi"); plt.ylabel("PR-AUC (CV)"); plt.title("Hasil tuning XGBoost: 20 kombinasi")
plt.grid(alpha=.3); savefig("05_xgb_tuning.png")

## 7. Model 3: Deep Learning (DNN / MLP dengan Keras)
Arsitektur: Dense (ReLU) + BatchNorm + Dropout, output 1 neuron sigmoid.
Ketimpangan kelas ditangani dengan `class_weight`. Training dihentikan otomatis (`EarlyStopping`) berdasarkan PR-AUC validation.
Kita coba beberapa konfigurasi (tuning sederhana) dan pilih yang PR-AUC validation-nya tertinggi.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
tf.keras.utils.set_random_seed(SEED)

def build_dnn(units, dropout, lr, n_in=30):
    m = keras.Sequential([layers.Input((n_in,))])
    for u in units:
        m.add(layers.Dense(u, activation="relu"))
        m.add(layers.BatchNormalization()); m.add(layers.Dropout(dropout))
    m.add(layers.Dense(1, activation="sigmoid"))
    m.compile(optimizer=keras.optimizers.Adam(lr), loss="binary_crossentropy",
              metrics=[keras.metrics.AUC(curve="PR", name="pr_auc")])
    return m

# class weight: fraud dibobot lebih besar (dibatasi supaya training stabil)
cw = {0: 1.0, 1: float(min(spw, 100.0))}
configs = [
    {"units": [32, 16], "dropout": 0.2, "lr": 1e-3},
    {"units": [64, 32], "dropout": 0.3, "lr": 1e-3},
    {"units": [128, 64, 32], "dropout": 0.3, "lr": 5e-4},
    {"units": [64, 32], "dropout": 0.2, "lr": 3e-3},
]
results, histories, models = [], [], []
for i, cfg in enumerate(configs):
    tf.keras.utils.set_random_seed(SEED)
    m = build_dnn(**cfg)
    es = keras.callbacks.EarlyStopping(monitor="val_pr_auc", mode="max", patience=8, restore_best_weights=True)
    h = m.fit(Xtr.values.astype("float32"), y_tr.values, validation_data=(Xva.values.astype("float32"), y_va.values),
              epochs=60, batch_size=1024, class_weight=cw, callbacks=[es], verbose=0)
    pva = m.predict(Xva.values.astype("float32"), verbose=0).ravel()
    score = average_precision_score(y_va, pva)
    results.append({**cfg, "units": str(cfg["units"]), "epochs_run": len(h.history["loss"]), "val_pr_auc": round(score, 4)})
    histories.append(h); models.append(m)
    print(f"config {i+1}/{len(configs)} {cfg} -> val PR-AUC {score:.4f}")
dnn_tab = pd.DataFrame(results); display(dnn_tab)
best_i = int(dnn_tab["val_pr_auc"].idxmax()); dnn = models[best_i]; hist = histories[best_i]
print("Konfigurasi DNN terbaik:", configs[best_i])
p_dnn_va = dnn.predict(Xva.values.astype("float32"), verbose=0).ravel()
p_dnn_te = dnn.predict(Xte.values.astype("float32"), verbose=0).ravel()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist.history["loss"], label="train"); ax[0].plot(hist.history["val_loss"], label="validation")
ax[0].set_title("Learning curve: loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(hist.history["pr_auc"], label="train"); ax[1].plot(hist.history["val_pr_auc"], label="validation")
ax[1].set_title("Learning curve: PR-AUC"); ax[1].set_xlabel("epoch"); ax[1].legend()
savefig("06_dnn_learning_curve.png")

## 8. Perbandingan model (data TEST)
Threshold tiap model dipilih dari data **validation** (F1 terbaik), lalu dipakai apa adanya di data test. Jadi test tetap "jujur".

In [ ]:
models_eval = {
    "XGBoost baseline": (p_base_va, p_base_te),
    "XGBoost tuned": (p_xt_va, p_xt_te),
    "DNN (Keras)": (p_dnn_va, p_dnn_te),
}
rows = []
thr_map = {}
for name, (pva, pte) in models_eval.items():
    thr = best_threshold(y_va, pva); thr_map[name] = thr
    rows.append(evaluate(name, y_te, pte, thr))
comp = pd.DataFrame(rows).set_index("model")
display(comp.round(4))
comp.round(4).to_csv(OUT / "model_comparison.csv")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
colors = ["#94a3b8", "#2dd4bf", "#f59e0b"]
for (name, (pva, pte)), c in zip(models_eval.items(), colors):
    p, r, _ = precision_recall_curve(y_te, pte); ax[0].plot(r, p, color=c, label=f"{name} (AP={average_precision_score(y_te, pte):.3f})")
    fpr, tpr, _ = roc_curve(y_te, pte); ax[1].plot(fpr, tpr, color=c, label=f"{name} (AUC={roc_auc_score(y_te, pte):.3f})")
ax[0].set_title("Precision-Recall curve (test)"); ax[0].set_xlabel("Recall"); ax[0].set_ylabel("Precision"); ax[0].legend()
ax[1].plot([0, 1], [0, 1], "k--", alpha=.4); ax[1].set_title("ROC curve (test)"); ax[1].set_xlabel("FPR"); ax[1].set_ylabel("TPR"); ax[1].legend()
comp[["precision", "recall", "f1", "pr_auc"]].plot.bar(ax=ax[2], rot=15, color=["#60a5fa", "#2dd4bf", "#f59e0b", "#a78bfa"])
ax[2].set_title("Perbandingan metrik (test)"); ax[2].set_ylim(0, 1.05)
savefig("07_model_comparison.png")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, (name, (pva, pte)) in zip(ax, models_eval.items()):
    cm = confusion_matrix(y_te, (pte >= thr_map[name]).astype(int))
    sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", cbar=False, ax=a,
                xticklabels=["Pred Normal", "Pred Fraud"], yticklabels=["Aktual Normal", "Aktual Fraud"])
    a.set_title(f"{name}\nthr={thr_map[name]:.3f}")
savefig("08_confusion_matrices.png")

## 9. Interpretasi: Feature importance & SHAP (XGBoost tuned)

In [ ]:
import shap
explainer = shap.TreeExplainer(xgb_tuned)
sample = Xte.sample(min(3000, len(Xte)), random_state=SEED)
sv = explainer.shap_values(sample)
shap.summary_plot(sv, sample, show=False, max_display=15)
savefig("09_shap_summary.png")
shap.summary_plot(sv, sample, plot_type="bar", show=False, max_display=15)
savefig("10_shap_bar.png")

## 10. Pilih model terbaik & export untuk API
Model dipilih berdasarkan **PR-AUC pada validation** (bukan test), supaya pemilihan tidak "mengintip" data test.
Hasil export masuk folder `artifacts_v2/`. Salin isinya ke folder `artifacts/` di repo GitHub kamu.

In [ ]:
val_scores = {n: average_precision_score(y_va, pva) for n, (pva, _) in models_eval.items()}
print("PR-AUC validation:", {k: round(v, 4) for k, v in val_scores.items()})
cand = {k: v for k, v in val_scores.items() if k != "XGBoost baseline"}   # baseline hanya pembanding
best_name = max(cand, key=cand.get)
print("Model terpilih:", best_name)

if best_name == "XGBoost tuned":
    xgb_tuned.get_booster().save_model(str(OUT / "model_xgboost.json"))
    kind, fname = "xgboost", "model_xgboost.json"
else:
    ws = dnn.get_weights()   # urutan: W,b tiap Dense; BatchNorm punya 4 parameter -> kita lipat ke Dense
    # Lipat BatchNorm ke Dense agar inferensi numpy sederhana (ReLU -> BN -> Dropout, jadi BN ada SETELAH ReLU).
    # Karena BN datang setelah ReLU, kita lipat BN ke Dense LAYER BERIKUTNYA.
    dense_layers = [l for l in dnn.layers if isinstance(l, layers.Dense)]
    bn_layers = [l for l in dnn.layers if isinstance(l, layers.BatchNormalization)]
    Ws = [l.get_weights()[0].astype("float64") for l in dense_layers]
    bs = [l.get_weights()[1].astype("float64") for l in dense_layers]
    for i, bn in enumerate(bn_layers):
        g, be, mu, var = [a.astype("float64") for a in bn.get_weights()]
        s = g / np.sqrt(var + bn.epsilon)        # h_bn = s*h + (be - mu*s)
        shift = be - mu * s
        # layer berikutnya: (s*h + shift) @ W + b = h @ (s[:,None]*W) + (shift @ W + b)
        bs[i + 1] = shift @ Ws[i + 1] + bs[i + 1]
        Ws[i + 1] = s[:, None] * Ws[i + 1]
    np.savez(OUT / "model_dnn.npz", **{f"W{i}": W for i, W in enumerate(Ws)}, **{f"b{i}": b for i, b in enumerate(bs)})
    kind, fname = "dnn", "model_dnn.npz"
    # verifikasi: inferensi numpy harus sama dengan Keras
    h = Xte.values.astype("float64")
    for W, b in zip(Ws[:-1], bs[:-1]): h = np.maximum(h @ W + b, 0)
    z = h @ Ws[-1] + bs[-1]; p_np = (1 / (1 + np.exp(-z))).ravel()
    print("Selisih maksimum numpy vs Keras:", float(np.abs(p_np - p_dnn_te).max()))
    assert np.abs(p_np - p_dnn_te).max() < 1e-4, "Export DNN tidak cocok dengan Keras!"

m = comp.loc[best_name]
meta = {"model_kind": kind, "model_file": fname, "version": "v2-" + best_name.lower().replace(" ", "-").replace("(", "").replace(")", ""),
        "threshold": float(thr_map[best_name]),
        "description": f"{best_name}, dipilih dari PR-AUC validation. Threshold dari F1 terbaik di validation.",
        "metrics_test": {k: (float(m[k]) if k in ("precision", "recall", "f1") else int(m[k]))
                         for k in ["precision", "recall", "f1", "tn", "fp", "fn", "tp"]}}
(OUT / "model_meta.json").write_text(json.dumps(meta, indent=2))
(OUT / "preprocessing.json").write_text(json.dumps(preproc, indent=2))
print(json.dumps(meta, indent=2))

In [ ]:
# Uji coba: file hasil export dimuat ulang & dibandingkan dengan model di memori
if kind == "xgboost":
    chk = xgb.Booster(); chk.load_model(str(OUT / "model_xgboost.json"))
    p_chk = chk.predict(xgb.DMatrix(Xte.values.astype("float32"), feature_names=FEATURES))
    print("Selisih maksimum:", float(np.abs(p_chk - p_xt_te).max()))
print("File di artifacts_v2:", sorted(p.name for p in OUT.iterdir()))

In [ ]:
# Unduh semuanya (artefak + gambar) dalam satu zip
import shutil
shutil.make_archive("hasil_v2", "zip", ".", "artifacts_v2")
shutil.make_archive("figures_v2", "zip", ".", "figures")
from google.colab import files
files.download("hasil_v2.zip"); files.download("figures_v2.zip")

## Selesai
1. Ekstrak `hasil_v2.zip` lalu salin 3 file ke folder `artifacts/` di repo (timpa yang lama).
2. Kirim ke aku: tabel perbandingan model (output sel 8) + `figures_v2.zip`, supaya slide PPT diisi dengan angka asli.